In [ ]:
!mkdir -p ./weights ./embedding_index  ./images

In [ ]:
!pip install -q torch torchvision

## Downloading model weights

In [ ]:
# !wget https://dl.fbaipublicfiles.com/ijepa/IN1K-vit.h.14-300e.pth.tar -O ./weights/IN1K-vit.h.14-300e.pth.tar

In [ ]:
# !wget https://dl.fbaipublicfiles.com/ijepa/IN1K-vit.h.16-448px-300e.pth.tar -O ./weights/IN1K-vit.h.16-448px-300e.pth.tar

In [ ]:
# !wget https://dl.fbaipublicfiles.com/ijepa/IN22K-vit.g.16-600e.pth.tar -O ./weights/IN22K-vit.g.16-600e.pth.tar

In [ ]:
!wget https://dl.fbaipublicfiles.com/ijepa/IN22K-vit.h.14-900e.pth.tar -O ./weights/IN22K-vit.h.14-900e.pth.tar

--2026-04-23 11:04:42--  https://dl.fbaipublicfiles.com/ijepa/IN22K-vit.h.14-900e.pth.tar
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.189.108, 3.163.189.14, 3.163.189.96, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.189.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10358004345 (9.6G) [application/x-tar]
Saving to: ‘./weights/IN22K-vit.h.14-900e.pth.tar’

./weights/IN22K-vit 100%[===================>]   9.65G  50.5MB/s    in 3m 25s  

2026-04-23 11:08:07 (48.2 MB/s) - ‘./weights/IN22K-vit.h.14-900e.pth.tar’ saved [10358004345/10358004345]



# Model

## Patch Embedding

In [ ]:
import torch
import torch.nn as nn
import math
from typing import Optional, Tuple, Union


class PatchEmbed(nn.Module):
  """Image to Patch Embedding"""
  def __init__(
      self,
      img_size: int = 224,
      patch_size: int = 14,
      in_chans: int = 3,
      embed_dim: int = 768,
  ):
    super().__init__()
    self.img_size = img_size
    self.patch_size = patch_size
    self.grid_size = img_size // patch_size
    self.num_patches = self.grid_size ** 2

    self.proj = nn.Conv2d(
        in_chans,
        embed_dim,
        kernel_size=patch_size,
        stride=patch_size
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # x: (B, C, H, W) -> (B, N, D)
    x = self.proj(x)  # (B, D, H/P, W/P)
    x = x.flatten(2)   # (B, D, H/P * W/P (N))
    x = x.transpose(1, 2)  # (B, N, D)
    return x

In [ ]:
temp_tensor = torch.randn(1, 3, 224, 224)
temp_patch_embed = PatchEmbed()
temp_img_patch = temp_patch_embed(temp_tensor)

print(temp_img_patch.shape)

torch.Size([1, 256, 768])


## Multi Headed Attention

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(
      self,
      dim: int,
      num_heads: int = 8,
      qkv_bias: bool = True,
      attn_drop: float = 0.0,
      proj_drop: float = 0.0,
  ):
    super().__init__()
    self.num_heads = num_heads
    head_dim = dim // num_heads
    self.scale = head_dim ** -0.5

    self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
    self.attn_drop = nn.Dropout(attn_drop)
    self.proj = nn.Linear(dim, dim)
    self.proj_drop = nn.Dropout(proj_drop)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    B, N, C = x.shape   # (1, 196, 768)
    qkv = self.qkv(x)   # (1, 196, 2304)  [768 * 3 = 2304]
    qkv = qkv.reshape(B, N, 3, self.num_heads, C // self.num_heads)   # (1, 196, 3, 8, 96)
    qkv = qkv.permute(2, 0, 3, 1, 4)    # (3, 1, 8, 196, 96)
    q, k, v = qkv[0], qkv[1], qkv[2]    # Each: (1, 8, 196, 96)

    attn = (q @ k.transpose(-2, -1))   # matrix mult (1, 8, 196, 196)
    attn = attn * self.scale   # scale to prevent softmax saturation
    attn = attn.softmax(dim=-1)  # normalize to probabilities
    attn = self.attn_drop(attn)  # regularization (1, 8, 196, 196)

    x = (attn @ v)   # Each head produces attended features (1, 8, 196, 96)
    x = x.transpose(1, 2)   # (1, 196, 8, 96)
    x = x.reshape(B, N, C)  # (1, 196, 768)
    x = self.proj(x)   # projection (1, 196, 768)
    x = self.proj_drop(x)   # dropout (1, 196, 768)
    return x


## Multi Layer Perceptron

In [ ]:
class MLP(nn.Module):
  def __init__(
      self,
      in_features: int,
      hidden_features: Optional[int] = None,
      out_features: Optional[int] = None,
      drop: float = 0.0,
  ):
    super().__init__()
    out_features = out_features or in_features
    hidden_features = hidden_features or in_features

    self.fc1 = nn.Linear(in_features, hidden_features)
    self.act = nn.GELU()
    self.fc2 = nn.Linear(hidden_features, out_features)
    self.drop = nn.Dropout(drop)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x = self.fc1(x)
    x = self.act(x)
    x = self.drop(x)
    x = self.fc2(x)
    x = self.drop(x)
    return x

## Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(
      self,
      dim: int,
      num_heads: int,
      mlp_ratio: float = 4.0,
      qkv_bias: bool = True,
      drop: float = 0.0,
      attn_drop: float = 0.0,
  ):
    super().__init__()
    self.norm1 = nn.LayerNorm(dim, eps=1e-6)
    self.attn = MultiHeadAttention(
        dim, num_heads=num_heads, qkv_bias=qkv_bias,
        attn_drop=attn_drop, proj_drop=drop
    )
    self.norm2 = nn.LayerNorm(dim, eps=1e-6)
    mlp_hidden_dim = int(dim * mlp_ratio)
    self.mlp = MLP(
        in_features=dim,
        hidden_features=mlp_hidden_dim,
        drop=drop
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x = x + self.attn(self.norm1(x))
    x = x + self.mlp(self.norm2(x))
    return x

## I-JEPA Target Encoder

In [ ]:
class IJEPATargetEncoder(nn.Module):
  """
  Standard ViT without classification head.
  Processes full image and outputs patch-level representations.
  """
  def __init__(
      self,
      img_size: int = 224,
      patch_size: int = 14,
      in_chans: int = 3,
      embed_dim: int = 768,
      depth: int = 12,
      num_heads: int = 12,
      mlp_ratio: float = 4.0,
      qkv_bias: bool = True,
      drop_rate: float = 0.0,
      attn_drop_rate: float = 0.0,
      norm_layer: Optional[nn.Module] = None,
  ):
    super().__init__()
    self.num_features = self.embed_dim = embed_dim
    norm_layer = norm_layer or nn.LayerNorm

    # Patch embedding
    self.patch_embed = PatchEmbed(
        img_size=img_size,
        patch_size=patch_size,
        in_chans=in_chans,
        embed_dim=embed_dim,
    )
    num_patches = self.patch_embed.num_patches

    # Positional embedding (learnable)
    self.pos_embed = nn.Parameter(
        torch.zeros(1, num_patches, embed_dim)
    )
    self.pos_drop = nn.Dropout(p=drop_rate)

    # Transformer blocks
    self.blocks = nn.ModuleList([
        TransformerBlock(
            dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias,
            drop=drop_rate,
            attn_drop=attn_drop_rate,
        )
        for _ in range(depth)
    ])

    self.norm = norm_layer(embed_dim, eps=1e-6)

    # Initialize weights
    nn.init.trunc_normal_(self.pos_embed, std=0.02)
    self.apply(self._init_weights)

  def _init_weights(self, m):
    if isinstance(m, nn.Linear):
      nn.init.trunc_normal_(m.weight, std=0.02)
      if m.bias is not None:
        nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.LayerNorm):
      nn.init.constant_(m.bias, 0)
      nn.init.constant_(m.weight, 1.0)

  def forward(
      self,
      x: torch.Tensor,
      return_all_tokens: bool = True,
      patch_indices: Optional[torch.Tensor] = None,
  ) -> torch.Tensor:
    """
    Args:
      x: input images (B, C, H, W)
      return_all_tokens: If true, return all patch tokens
      patch_indices: If provided, return only specific patch indices

    Returns:
      Patch representations (B, N, D) or (B, len(indices), D)
    """
    # Patch embedding
    x = self.patch_embed(x)   # (B, N, D)

    # Add positional embeddings
    x = x + self.pos_embed
    x = self.pos_drop(x)

    # Apply transformer blocks
    for block in self.blocks:
      x = block(x)

    x = self.norm(x)

    # Return specific patches if indices provided
    if patch_indices is not None:
      x = x[:, patch_indices, :]

    return x

  def get_layer_representations(
      self,
      x: torch.Tensor,
      strategy: str = "last",
      specific_indices: Optional[list[int]] = None,
      patch_indices: Optional[torch.Tensor] = None,
  ) -> torch.Tensor:
    """
    Extract semantic representations using different layer strategies.

    Args:
      x: Input images (B, C, H, W)
      strategy: Strategy to extract representations
        - 'last': final layer only (baseline)
        - 'second-last': 2nd-to-last block output
        - 'last_four_concat': concat of last 4 layer (B, N, 4*D)
        - 'specific': layers at specific_indices (eg: [25,27,29,31])
      specific_indices: Block indices to use when strategy='specific'
      patch_indices: If provided, return only these patch posistions

    Returns:
      (B, N, D) for "last"/"second_last", (B, N, 4*D) for concat strategies
    """
    x = self.patch_embed(x)
    x = x + self.pos_embed
    x = self.pos_drop(x)

    n_blocks = len(self.blocks)

    # Determnine which block indices to capture
    if strategy == "second_last":
      capture_at = {n_blocks - 2}
    elif strategy == "last_four_concat":
      capture_at = set(range(n_blocks - 4, n_blocks))
    elif strategy == "specific":
      assert specific_indices is not None, "Provide specific_indices when strategy='specific'"
      capture_at = set(specific_indices)
    else:
      capture_at = {n_blocks - 1}

    captured = {}
    for i, block in enumerate(self.blocks):
      x = block(x)
      if i in capture_at:
        captured[i] = x.clone()

    # Apply norm and pool
    if strategy in ("last_four_concat", "specific"):
      # Sort by layer order, normalize each, then concat along D
      layers = [self.norm(captured[i]) for i in sorted(captured)]
      out = torch.cat(layers, dim=-1)   # (B, N, num_layers * D)
    elif strategy == "second_last":
      out = self.norm(captured[n_blocks - 2])
    else:
      out = self.norm(x)    # x is already the last block output

    if patch_indices is not None:
      out = out[:, patch_indices, :]

    return out


## Model Configurations matching I-JEPA paper

In [ ]:
def vit_huge(patch_size=14, **kwargs):
    """ViT-Huge (ViT-H/14 or ViT-H/16) - embed_dim=1280, depth=32, num_heads=16"""
    print("   1.1 ViT-Huge ...")
    model = IJEPATargetEncoder(
        patch_size=patch_size,
        embed_dim=1280,
        depth=32,
        num_heads=16,
        mlp_ratio=4,
        **kwargs
    )
    return model

def vit_giant(patch_size=16, **kwargs):
    """ViT-Giant (ViT-G/16) - embed_dim=1408, depth=40, num_heads=16"""
    model = IJEPATargetEncoder(
        patch_size=patch_size,
        embed_dim=1408,
        depth=40,
        num_heads=16,
        mlp_ratio=48/11,  # ~4.36
        **kwargs
    )
    return model

# Embedding

## Embedding Generator

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import pickle
from pathlib import Path
from typing import List, Union, Tuple, Optional
import tqdm


class ImageEmbeddingDataset(Dataset):
  """Dataset for batch image embedding generation"""
  def __init__(
      self,
      image_paths: List[Union[str, Path]],
      transform=None
  ):
    self.image_paths = [Path(p) for p in image_paths]
    self.transform = transform or self.default_transform()

  @staticmethod
  def default_transform():
    # I-JEPA uses mean=05 and std=0.5 normalization
    return transforms.Compose([
        transforms.Resize(224,
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    img_path = self.image_paths[idx]
    image = Image.open(img_path).convert('RGB')
    image = self.transform(image)
    return image, str(img_path)


class EmbeddingGenerator:
  """Generate embeddings for image database using batch inference."""
  def __init__(
      self,
      model: nn.Module,
      device: str = "cuda" if torch.cuda.is_available() else "cpu",
      batch_size: int = 4,
      num_workers: int = 1,
      layer_strategy: str = "second_last",
      specific_indices: Optional[List[int]] = None,
  ):
    self.model = model.to(device).eval()
    self.device = device
    self.batch_size = batch_size
    self.num_workers = num_workers
    self.layer_strategy = layer_strategy
    self.specific_indices = specific_indices

    # Freeze model
    for param in self.model.parameters():
      param.requires_grad = False

  def _get_features(self, images: torch.Tensor) -> torch.Tensor:
    """Central routing method - all forward calss go through here."""
    if self.layer_strategy == "last":
      return self.model(images)
    return self.model.get_layer_representations(
        images,
        strategy=self.layer_strategy,
        specific_indices=self.specific_indices,
    )

  @torch.no_grad()
  def generate_embeddings(
      self,
      image_paths: List[Union[str, Path]],
      return_paths: bool = True,
      show_progress: bool = True,
  ) -> Union[np.ndarray, Tuple[np.ndarray, List[str]]]:
    """
    Generate embeddings for all images.

    Returns:
      embeddings: (N, D) array of embeddings
      paths: (optional) list of image paths
    """
    print("   3.1 Image Embedding Dataset...")
    dataset = ImageEmbeddingDataset(image_paths)
    print("   3.2 DataLoader...")
    dataloader = DataLoader(
        dataset,
        batch_size=self.batch_size,
        shuffle=False,
        num_workers=self.num_workers,
        pin_memory=True,
    )

    all_embeddings = []
    all_paths = []

    print("   3.3 tqdm iterator...\n")
    iterator = tqdm.tqdm(dataloader, desc="Generating embeddings") if show_progress else dataloader

    print("   3.4 for loop...")
    for batch_images, batch_paths in iterator:
      batch_images = batch_images.to(self.device, non_blocking=True)

      # Get embeddings: average pool patch tokens for global representation
      features = self._get_features(batch_images)   # (B, N, D)
      embeddings = features.mean(dim=1)     # (B, D)

      # L2 normalization for cosine similarity
      embeddings = nn.functional.normalize(embeddings, p=2, dim=1)

      all_embeddings.append(embeddings.cpu().numpy())
      all_paths.extend(batch_paths)

    embeddings = np.vstack(all_embeddings)

    if return_paths:
      return embeddings, all_paths
    return embeddings

  def generate_single_embedding(
      self, image: Union[str, Path, Image.Image, torch.Tensor]
  ) -> np.ndarray:
    """Generate embedding for a single image"""
    transform = ImageEmbeddingDataset.default_transform()

    if isinstance(image, (str, Path)):
      image = Image.open(image).convert('RGB')

    if isinstance(image, Image.Image):
      image = transform(image)

    if isinstance(image, torch.Tensor):
      image = image.unsqueeze(0) if image.dim() == 3 else image

    image = image.to(self.device)

    with torch.no_grad():
      features = self._get_features(image)
      embedding = features.mean(dim=1)
      embedding = nn.functional.normalize(embedding, p=2, dim=1)

    return embedding.cpu().numpy()

## Similarity Search Index

### FAISS

In [ ]:
# !pip install -q faiss-gpu-cu12

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.5 MB/s eta 0:00:00


In [ ]:
import faiss

class ImageSimilaritySearch:
  """FAISS-based similarity search for image embeddings"""

  def __init__(
      self,
      dimension: int = 768,  # Dependes on model size (768 for ViT-B)
      index_type: str = "cosine",  # "cosine", "l2", "ip" (inner product)
      use_gpu: bool = False,
  ):
    self.dimension = dimension
    self.index_type = index_type
    self.use_gpu = use_gpu
    self.index = None
    self.image_paths = []
    self.metadata = {}

    self._create_index()

  def _create_index(self):
    """Create FAISS index based on similarity metric"""
    if self.index_type == "cosine":
      # For cosine similarity, we use IndexFlatIP with normalized vectors
      # vectors are already L2 normalized in embedding generator
      self.index = faiss.IndexFlatIP(self.dimension)
    elif self.index_type == "l2":
      self.index = faiss.IndexFlatL2(self.dimension)
    elif self.index_type == "ip":
      self.index = faiss.IndexFlatIP(self.dimension)
    else:
      raise ValueError(f"Unknown index type: {self.index_type}")

    # Move to GPU if requested
    if self.use_gpu and faiss.get_num_gpus() > 0:
      res = faiss.StandardGpuResources()
      self.index = faiss.index_cpu_to_gpu(res, 0, self.index)

    print(f"Created {self.index_type} index (dim={self.dimension})")

  def add_embeddings(
      self,
      embeddings: np.ndarray,
      image_paths: List[str],
      metadata: Optional[dict] = None,
  ):
    """
    Add embeddings to the index.

    Args:
      embeddings: (N, D) array of L2-normalized embeddings
      image_paths: List of N image paths
      metadata: Optional dict with additional info for each image
    """
    assert len(embeddings) == len(image_paths), "Embeddings and paths must match"
    assert embeddings.shape[1] == self.dimension, f"Expected dim {self.dimension}, got {embeddings.shape[1]}"

    # Ensure float32 and contiguous
    embeddings = np.ascontiguousarray(embeddings.astype('float32'))

    # Add to index
    start_idx = len(self.image_paths)
    self.index.add(embeddings)
    self.image_paths.extend(image_paths)

    # Store metadata
    if metadata:
      for i, path in enumerate(image_paths):
        self.metadata[path] = metadata.get(start_idx + i, {})

    print(f"Added {len(embeddings)} embeddings. Total: {len(self.image_paths)}")

  def search(
      self,
      query_embedding: np.ndarray,
      k: int = 5,
      return_scores: bool = True,
  ) -> Union[List[str], List[Tuple[str, float]]]:
    """
    Search for k most similar images.

    Args:
      query_embeddings: (1, D) or (D,) array - single query embedding
      k: Number of results to return
      return_scores: If True, return (path, score) tuples

    Returns:
      List of image paths or (path, similarity_score) tuple
    """
    if query_embedding.ndim == 1:
      query_embedding = query_embedding.reshape(1, -1)

    # Ensure float32 and contiguous
    query_embedding = np.ascontiguousarray(query_embedding.astype('float32'))

    # Search
    scores, indices = self.index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
      if idx == -1:   # FAISS returns -1 for not enough results
        continue
      path = self.image_paths[idx]
      if return_scores:
        results.append((path, float(score)))
      else:
        results.append(path)

    return results

  def batch_search(
      self,
      query_embeddings: np.ndarray,
      k: int = 5,
  ) -> List[List[Tuple[str, float]]]:
    """Search for multiple queries at once."""
    query_embeddings = np.ascontiguousarray(
        query_embeddings.astype('float32')
    )
    scores, indices = self.index.search(query_embeddings, k)

    all_results = []
    for query_scores, query_indices in zip(scores, indices):
      results = []
      for score, idx in zip(query_scores, query_indices):
        if idx == -1:
          continue
        results.append((self.image_paths[idx], float(score)))
      all_results.append(results)

    return all_results

  def save(self, save_dir: Union[str, Path]):
    """Save index and metadata."""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # Save FAISS index
    index_path = save_dir / "index.faiss"
    # Move to CPU before saving if on GPU
    if self.use_gpu:
      cpu_index = faiss.index_gpu_to_cpu(self.index)
      faiss.write_index(cpu_index, str(index_path))
    else:
      faiss.write_index(self.index, str(index_path))

    # Save metadata
    metadata = {
        'image_paths': self.image_paths,
        'metadata': self.metadata,
        'dimension': self.dimension,
        'index_type': self.index_type,
    }
    with open(save_dir / "metadata.pkl", 'wb') as f:
      pickle.dump(metadata, f)

    print(f"Saved index to {save_dir}")

  @classmethod
  def load(cls, save_dir: Union[str, Path], use_gpu: bool = False):
    """Load index and metadata."""
    save_dir = Path(save_dir)

    # Load metadata
    with open(save_dir / "metadata.pkl", 'rb') as f:
      metadata = pickle.load(f)

    # Create instance
    instance = cls(
        dimension=metadata['dimension'],
        index_type=metadata['index_type'],
        use_gpu=use_gpu,
    )

    # Load FAISS index
    index_path = save_dir / "index.faiss"
    instance.index = faiss.read_index(str(index_path))

    if use_gpu and faiss.get_num_gpus() > 0:
      res = faiss.StandardGpuResources()
      instance.index = faiss.index_cpu_to_gpus(res, 0, instance.index)

    instance.image_paths = metadata['image_paths']
    instance.metadata = metadata['metadata']

    print(f"Loaded index with {len(instance.image_paths)} embeddings")
    return instance


### Pinecone

In [ ]:
!pip install -q pinecone

In [ ]:
import uuid
import numpy as np
from pathlib import Path
from google.colab import userdata
from typing import List, Union, Tuple, Optional


class PineconeImageSimilaritySearch:
    """
    Pinecone-backend similarity search for image embeddings. Image paths are stored as
    Pinecone vector metadata so no local state is needed between sessions.
    """

    def __init__(
        self,
        index_name: str,
        api_key: str,
        cloud: str,
        region: str,
        dimension: int = 5120,
        namespace: str = "",
        metric: str = "cosine",

        create_if_missing: bool = True,
    ):
        try:
            from pinecone import Pinecone, ServerlessSpec
        except ImportError:
            raise ImportError("Install Pinecone first")

        self.dimension = dimension
        self.index_type = metric
        self.namespace = namespace
        self._index_name = index_name
        self.image_paths: List[str] = []
        self.metadata: dict = {}

        pc = Pinecone(api_key=api_key)

        existing = [idx.name for idx in pc.list_indexes()]
        if index_name not in existing:
            if not create_if_missing:
                raise ValueError(
                    f"Index '{index_name}' not found and create_if_missing=False."
                )
            pc.create_index(
                name=index_name,
                dimension=dimension,
                metric=metric,
                spec=ServerlessSpec(cloud=cloud, region=region),
            )
            print(f"Created Pinecone index '{index_name}' ({metric}, dim={dimension})")
        else:
            print(f"Connected to existing Pinecone index '{index_name}'")

        self.index = pc.Index(index_name)

    def add_embeddings(
        self,
        embeddings: np.ndarray,
        image_paths: List[str],
        metadata: Optional[dict] = None,
    ):
        """
        Upsert embeddings into Pinecone.

        Args:
            embeddings: (N, D) float32 array for L2-normalised embeddings
            image_paths: List of N image paths (stored as Pinecone metadata)
            metadata: Optional {int_index: dict} of extra per-image metadata
        """
        assert len(embeddings) == len(image_paths), "Embeddings and paths must match"
        assert embeddings.shape[1] == self.dimension, (
            f"Expected dim {self.dimension}, got {embeddings.shape[1]}"
        )

        embeddings = embeddings.astype("float32")

        vectors = []
        for i, (emb, path) in enumerate(zip(embeddings, image_paths)):
            vec_id = str(uuid.uuid4())
            meta = {"image_path": path}
            if metadata and i in metadata:
                meta.update(metadata[i])
            vectors.append({"id": vec_id, "values": emb.tolist(), "metadata": meta})

        # Pinecone recommends batches of <= 100
        batch_size = 100
        for start in range(0, len(vectors), batch_size):
            self.index.upsert(
                vectors=vectors[start: start + batch_size],
                namespace=self.namespace,
            )

        self.image_paths.extend(image_paths)
        if metadata:
            for i, path in enumerate(image_paths):
                if i in metadata:
                    self.metadata[path] = metadata[i]

        print(f"Upserted {len(embeddings)} vectors. "
              f"Total (local cache): {len(self.image_paths)}")

    def search(
        self,
        query_embedding: np.ndarray,
        k: int = 5,
        return_scores: bool = True,
        filter: Optional[dict] = None,
    ) -> Union[List[str], List[Tuple[str, float]]]:
        """
        Search for the k most similar images.

        Args:
            query_embedding: (1, D) or (D,) float32 array
            k: Number of results
            return_scores: If True, return (path, score) tuples
            filter: Optional Pinecone metadata filter dict
        """
        if query_embedding.ndim == 1:
            query_embedding = query_embedding.reshape(1, -1)

        query_list = query_embedding[0].astype("float32").tolist()

        kwargs = dict(
            vector=query_list,
            top_k=k,
            include_metadata=True,
            namespace=self.namespace,
        )
        if filter:
            kwargs["filter"] = filter

        response = self.index.query(**kwargs)

        results = []
        for match in response.matches:
            path = match.metadata.get("image_path", match.id)
            if return_scores:
                results.append((path, float(match.score)))
            else:
                results.append(path)

        return results

    def batch_search(
        self,
        query_embeddings: np.ndarray,
        k: int = 5,
        filter: Optional[dict] = None,
    ) -> List[List[Tuple[str, float]]]:
        """Search for multiple query embeddings sequentially."""
        return [
            self.search(q, k=k, return_scores=True, filter=filter)
            for q in query_embeddings
        ]

    def save(self, save_dir: Union[str, Path]):
        """
        Pinecone vectors are already persisted server-side.
        This optionally saves the local image_path cache to disk so we don't
        have to re-scane the index on startup.
        """
        import pickle

        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        local_state = {
            "image_paths": self.image_paths,
            "metadata": self.metadata,
            "dimension": self.dimension,
            "namespace": self.namespace,
            "index_name": self._index_name,
            "metric": self.index_type,
        }
        with open(save_dir / "pinecone_local_cache.pkl", "wb") as f:
            pickle.dump(local_state, f)
        print(f"Saved local state to: {save_dir}")

    @classmethod
    def load(
        cls, save_dir: Union[str, Path], api_key: str, use_gpu: bool = False
    ) -> "PineconeImageSimilaritySearch":
        """
        Load local state and return a new PineconeImageSimilaritySearch instance.
        """
        import pickle

        save_dir = Path(save_dir)
        with open(save_dir / "pinecone_local_cache.pkl", "rb") as f:
            local_state = pickle.load(f)

        instance = cls(
            index_name=local_state["index_name"],
            api_key=api_key,
            dimension=local_state["dimension"],
            namespace=local_state["namespace"],
            metric=local_state["index_type"],
            create_if_mission=False,
        )
        instance.image_paths = local_state["image_paths"]
        instance.metadata = local_state["metadata"]
        return instance


# Save model

In [ ]:
import torch
import json
from dataclasses import dataclass, asdict

@dataclass
class ViTConfig:
    img_size: int = 224
    patch_size: int = 14
    in_chans: int = 3
    embed_dim: int = 1280
    depth: int = 32
    num_heads: int = 16
    mlp_ratio: float = 4.0

def save_model_package(
    model: torch.nn.Module,
    config: ViTConfig,
    save_dir: str = "weights/ijepa_model_package"
):
    """
    Save model weights + config separately (smaller than full model).
    """
    import os
    os.makedirs(save_dir, exist_ok=True)

    # Save config as JSON
    config_path = os.path.join(save_dir, "config.json")
    with open(config_path, 'w') as f:
        json.dump(asdict(config), f, indent=2)

    # Save weights only
    weights_path = os.path.join(save_dir, "model_weights.pth")
    torch.save(model.state_dict(), weights_path)

    # Check sizes
    weights_size = os.path.getsize(weights_path) / 1e9
    print(f"Model package saved to: {save_dir}")
    print(f"Weights size: {weights_size:.2f} GB")
    print(f"Config: {config_path}")

    return save_dir

def load_model_package(
    package_dir: str = "weights/ijepa_model_package",
    device: str = "cuda"
):
    """
    Load model from saved package (weights + config).
    """
    import os

    # Load config
    config_path = os.path.join(package_dir, "config.json")
    with open(config_path, 'r') as f:
        config_dict = json.load(f)
    config = ViTConfig(**config_dict)

    # Create model with correct architecture
    model = IJEPATargetEncoder(
        img_size=config.img_size,
        patch_size=config.patch_size,
        embed_dim=config.embed_dim,
        depth=config.depth,
        num_heads=config.num_heads,
        mlp_ratio=config.mlp_ratio
    )

    # Load weights
    weights_path = os.path.join(package_dir, "model_weights.pth")
    state_dict = torch.load(weights_path, map_location='cpu')
    model.load_state_dict(state_dict)

    # Move to device and set to eval
    # model = model.half().to(device).eval()
    model = model.to(device).eval()

    return model

# Complete Pipeline

## Before Saving model

In [ ]:
# Configuration
IMAGE_DIR = Path("./images")
INDEX_SAVE_DIR = Path("./embedding_index")
# CHECKPOINT_PATH = Path("./weights/IN22K-vit.h.14-300e.pth.tar")
CHECKPOINT_PATH = Path("./weights/IN22K-vit.h.14-900e.pth.tar")
# CHECKPOINT_PATH = Path("./weights/IN1K-vit.h.16-448px-300e.pth.tar")
# CHECKPOINT_PATH = Path("./weights/IN22K-vit.g.16-600e.pth.tar")

# Step 1: Initialize model
print("1. Loading I-JEPA model...")
# model = vit_giant(patch_size=16, img_size=224)
model = vit_huge(patch_size=14, img_size=224)

1. Loading I-JEPA model...
   1.1 ViT-Huge ...


In [ ]:
# Load pre-trained weights
print("   1.2 Loading pre-trained weights...")
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cuda:0')
# checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')

   1.2 Loading pre-trained weights...


In [ ]:
print("   1.3 Getting target_encoder weights...")
state_dict = checkpoint.get('target_encoder', checkpoint)

   1.3 Getting target_encoder weights...


In [ ]:
print("   1.4 Free up memory")
del checkpoint
import gc
gc.collect()
torch.cuda.empty_cache()
state_dict = {
    k.replace('module.', ''): v for k, v in
    state_dict.items()
}

   1.4 Free up memory


In [ ]:
print("   1.5 Loading state_dict in model...")
model.load_state_dict(state_dict, strict=False)
model = model.to("cuda")
model.eval()

   1.5 Loading state_dict in model...


IJEPATargetEncoder(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0-31): 32 x TransformerBlock(
      (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (qkv): Linear(in_features=1280, out_features=3840, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1280, out_features=1280, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=1280, out_features=5120, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=5120, out_features=1280, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
)

## Optional model saving and loading

In [ ]:
# Step 1.6: Downloading target encoder (Optional)

config = ViTConfig(img_size=224, patch_size=14, embed_dim=1280, depth=32, num_heads=16)
save_model_package(model, config, "weights/ijepa_vit_huge_package")

Model package saved to: weights/ijepa_vit_huge_package
Weights size: 2.52 GB
Config: weights/ijepa_vit_huge_package/config.json


'weights/ijepa_vit_huge_package'

In [ ]:
print("   1.6 Free up memory by deleting model")
del model
gc.collect()
torch.cuda.empty_cache()

   1.6 Free up memory by deleting model


In [ ]:
# Later - load easily
model = load_model_package("weights/ijepa_vit_huge_package")

## Rest of the steps

In [ ]:
# Step 2: Create embedding generator
print("2. Creating embedding generator...")
generator = EmbeddingGenerator(
    model=model,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=4,
    num_workers=1,
    layer_strategy="last_four_concat",
    # specific_indices=[25,27,29,31],
)

2. Creating embedding generator...


In [ ]:
# Step 3: Generate embeddings for all images
print("3. Generating embeddings for image database...")
image_paths = list(IMAGE_DIR.glob("*.jpg")) + list(IMAGE_DIR.glob("*.png"))
embeddings, paths = generator.generate_embeddings(image_paths)

3. Generating embeddings for image database...
   3.1 Image Embedding Dataset...
   3.2 DataLoader...
   3.3 tqdm iterator...



Generating embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

   3.4 for loop...


Generating embeddings: 100%|██████████| 5/5 [00:03<00:00,  1.32it/s]


In [ ]:
# Step 4: Create and populate similarity search index
print("4. Building similarity search index...")
# searcher = ImageSimilaritySearch(
#     dimension=5120,
#     index_type="cosine",
#     use_gpu=True,
# )
searcher = PineconeImageSimilaritySearch(
    index_name="ijepa-index",
    api_key=userdata.get("PINECONE"),
    dimension=5120,
    metric="cosine",
)
searcher.add_embeddings(embeddings, paths)

# Save index for later use
searcher.save(INDEX_SAVE_DIR)

4. Building similarity search index...
Created Pinecone index 'ijepa-index' (cosine, dim=5120)
Upserted 20 vectors. Total (local cache): 20
Saved local state to: embedding_index


In [ ]:
# Step 5: Query with new image
print("\n5. Performing similarity search...")
query_image_path = Path("./query_image_6.png")
query_embedding = generator.generate_single_embedding(query_image_path)

# Search
results = searcher.search(query_embedding, k=5, return_scores=True)

print(f"\nTop 5 similar images to {query_image_path}:")
for rank, (path, score) in enumerate(results, 1):
  print(f" {rank}. {path} (similarty: {score:.4f})")


5. Performing similarity search...

Top 5 similar images to query_image_6.png:
 1. images/silence.png (similarty: 0.8026)
 2. images/speed_homeless.png (similarty: 0.7728)
 3. images/druski_got_me.png (similarty: 0.7587)
 4. images/absolute_cinema.png (similarty: 0.7530)
 5. images/think_mark.png (similarty: 0.7477)
